Import libraries

In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from dragonnet import Dragonnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/Tarnet/main_tarnet.ipynb")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/train_women.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/val_women.csv")

CPU times: user 27.1 ms, sys: 8 ms, total: 35.1 ms
Wall time: 38.7 ms
CPU times: user 12 ms, sys: 4.92 ms, total: 16.9 ms
Wall time: 18.6 ms
CPU times: user 3.88 ms, sys: 2.82 ms, total: 6.7 ms
Wall time: 7.63 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.78953321  1.6460815   1.41745802 -1.11692492  0.91314538  1.07643872
   0.9950542  -0.36894675 -0.88924565  1.13133938]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25615, 10]); y=torch.Size([25615, 1]); t=torch.Size([25615, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [8]:
epochs = 150
lr = 5e-5
wd = 1e-5
alpha = 0.1
beta = 0.1
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
shared_dropout = 0
outcome_dropout = 0.0
shared_hidden = 200
outcome_hidden = 100
early_stop_start = 30
activation = torch.nn.ReLU
print (f" epochs = {epochs}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 learning rate = 5e-05
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
 early stop start = 30


In [9]:
import pandas as pd
import numpy as np
import torch
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True)
    grid_alpha = trial.suggest_categorical("alpha", [0.1, 0.5, 1.0])
    grid_beta = trial.suggest_categorical("beta", [0.1, 0.5, 1.0])

    val_scores = []
    for SEED in seeds:
        seed_everything(SEED)

        dragonnet = Dragonnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            alpha=grid_alpha,
            beta=grid_beta,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_droupout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Silence Dragonnet epoch logs
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            dragonnet.fit(train_loader, val_loader)

        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p, _, _ = dragonnet.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)
        val_scores.append(current_val_auqc)

    mean_val_auqc = float(np.mean(val_scores))
    std_val_auqc = float(np.std(val_scores, ddof=1)) if len(val_scores) > 1 else 0.0
    trial.set_user_attr("std_Val_AUQC", std_val_auqc)
    return mean_val_auqc

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# 2. Persist search results in notebook variables
trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": t.params["lr"],
        "weight_decay": t.params["weight_decay"],
        "alpha": t.params["alpha"],
        "beta": t.params["beta"],
        "mean_Val_AUQC": float(t.value),
        "std_Val_AUQC": float(t.user_attrs.get("std_Val_AUQC", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_Val_AUQC", ascending=False).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "alpha": float(best_params["alpha"]),
    "beta": float(best_params["beta"]),
    "mean_Val_AUQC": float(study.best_value),
    "std_Val_AUQC": float(study.best_trial.user_attrs.get("std_Val_AUQC", np.nan))
})

/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/50 [00:00<?, ?it/s]

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
Best trial: 41. Best value: 1.24957: 100%|██████████| 50/50 [52:14<00:00, 62.69s/it]


In [10]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 41
Best mean_Val_AUQC: 1.249571
Best params: {'lr': 0.00020739654007856528, 'weight_decay': 1.8037651472881755e-05, 'alpha': 0.5, 'beta': 0.5}

Best config table:


,lr,weight_decay,alpha,beta,mean_Val_AUQC,std_Val_AUQC
0,0.000207,0.000018,0.5,0.5,1.249571,0.102022



Top 10 trials:


,trial,lr,weight_decay,alpha,beta,mean_Val_AUQC,std_Val_AUQC
0,41,0.000207,0.000018,0.5,0.5,1.249571,0.102022
1,44,0.000200,0.000022,0.5,0.5,1.248457,0.100393
2,45,0.000205,0.000021,0.5,0.5,1.248221,0.104632
3,48,0.000275,0.000021,0.5,0.5,1.247192,0.111795
4,6,0.000184,0.000032,0.5,0.5,1.243448,0.094391
5,13,0.000217,0.000030,0.5,0.5,1.242827,0.102609
6,32,0.000169,0.000034,0.5,0.5,1.240855,0.094192
7,42,0.000193,0.000019,0.5,0.5,1.240567,0.093979
8,25,0.000255,0.000027,0.5,0.5,1.239930,0.092955
9,17,0.000214,0.000015,0.5,0.5,1.238195,0.115519


In [11]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_alpha = float(best_cfg['alpha'])
best_beta = float(best_cfg['beta'])

print("Đang đánh giá trên test với best config từ validation:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}, alpha={best_alpha:.1f}, beta={best_beta:.1f}")
print(f"Số seed: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    dragonnet = Dragonnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        alpha=best_alpha,
        beta=best_beta,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=shared_hidden,
        outcome_hidden=outcome_hidden,
        outcome_droupout=outcome_dropout,
        shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        activation=activation
    )

    dragonnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred, _, _ = dragonnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'CHI TIẾT TỪNG SEED (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'KẾT QUẢ TRUNG BÌNH TEST (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Đang đánh giá trên test với best config từ validation:
  lr=2.1e-04, weight_decay=1.8e-05, alpha=0.5, beta=0.5
Số seed: 5
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Base Loss: 0.3501 | Tarreg Loss: 0.763453 | Total Loss: 1.1136 | Val Loss: 194.1391 | Raw Qini: 1.3494 | EMA Trend: 1.3494 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 0.3365 | Tarreg Loss: 0.707950 | Total Loss: 1.0445 | Val Loss: 194.0579 | Raw Qini: 1.1924 | EMA Trend: 1.3259 | (patience: 1/20)
Epoch 3/150 | Base Loss: 0.3614 | Tarreg Loss: 0.773683 | Total Loss: 1.1351 | Val Loss: 193.9573 | Raw Qini: 1.1626 | EMA Trend: 1.3014 | (patience: 2/20)
Epoch 4/150 | Base Loss: 0.4026 | Tarreg Loss: 0.908499 | Total Loss: 1.3111 | Val Loss: 193.8406 | Raw Qini: 1.15

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 0.3714 | Tarreg Loss: 0.447824 | Total Loss: 0.8192 | Val Loss: 193.8441 | Raw Qini: 0.2947 | EMA Trend: 0.2947 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 0.3884 | Tarreg Loss: 0.442558 | Total Loss: 0.8310 | Val Loss: 193.7732 | Raw Qini: 0.2381 | EMA Trend: 0.2862 | (patience: 1/20)
Epoch 3/150 | Base Loss: 0.4131 | Tarreg Loss: 0.436588 | Total Loss: 0.8497 | Val Loss: 193.6941 | Raw Qini: 0.2558 | EMA Trend: 0.2816 | (patience: 2/20)
Epoch 4/150 | Base Loss: 229.1358 | Tarreg Loss: 40.480015 | Total Loss: 269.6159 | Val Loss: 193.5964 | Raw Qini: 0.0229 | EMA Trend: 0.2428 | (patience: 3/20)
Epoch 5/150 | Base Loss: 0.5239 | Tarreg Loss: 0.590149 | Total Loss: 1.1140 | Val Loss: 193.4664 | Raw Qini: 0.0496 | EMA Trend: 0.2138 | (patience: 4/20)
Epoch 6/150 | Base Loss: 0.5887 | Tarreg Loss: 0.399816 | Total Loss: 0.9885 | Val Loss: 193.3210 | Raw Qini: 0.2454 | EMA Trend: 0.2186 | ✓ above trend but not peak (patience: 5/20)
Epoch 7/150 | Base Loss

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 0.3527 | Tarreg Loss: 4.626943 | Total Loss: 4.9797 | Val Loss: 194.0078 | Raw Qini: 0.7407 | EMA Trend: 0.7407 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 0.3602 | Tarreg Loss: 4.606780 | Total Loss: 4.9670 | Val Loss: 193.9513 | Raw Qini: 0.7323 | EMA Trend: 0.7394 | (patience: 1/20)
Epoch 3/150 | Base Loss: 0.3752 | Tarreg Loss: 4.730732 | Total Loss: 5.1059 | Val Loss: 193.8871 | Raw Qini: 0.7488 | EMA Trend: 0.7408 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 0.3970 | Tarreg Loss: 4.384836 | Total Loss: 4.7818 | Val Loss: 193.8118 | Raw Qini: 0.7907 | EMA Trend: 0.7483 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 0.4597 | Tarreg Loss: 4.417722 | Total Loss: 4.8775 | Val Loss: 193.7240 | Raw Qini: 0.7815 | EMA Trend: 0.7533 | ✓ above trend but not peak (patience: 1/20)
Epoch 6/150 | Base Loss: 0.5271 | Tarreg Loss: 3.718737 | Total Loss: 4.2458 | Val Loss: 193.6305 | Raw Qini: 0.7703 | EMA Trend: 0.7558 | ✓ above trend but not peak 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 0.3740 | Tarreg Loss: 1.737788 | Total Loss: 2.1118 | Val Loss: 193.8739 | Raw Qini: 0.4637 | EMA Trend: 0.4637 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 0.3771 | Tarreg Loss: 1.613935 | Total Loss: 1.9910 | Val Loss: 193.8109 | Raw Qini: 0.7517 | EMA Trend: 0.5069 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Base Loss: 0.3988 | Tarreg Loss: 1.570119 | Total Loss: 1.9689 | Val Loss: 193.7367 | Raw Qini: 0.8945 | EMA Trend: 0.5650 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Base Loss: 0.4258 | Tarreg Loss: 1.344775 | Total Loss: 1.7706 | Val Loss: 193.6411 | Raw Qini: 0.9827 | EMA Trend: 0.6277 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Base Loss: 0.5021 | Tarreg Loss: 1.611344 | Total Loss: 2.1135 | Val Loss: 193.5225 | Raw Qini: 1.0334 | EMA Trend: 0.6886 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Base Loss: 0.6068 | Tarreg Loss: 1.307186 | Total Loss: 1.9139 | Val Loss: 193.3985 | Raw Qini: 1.0418 | EMA Trend: 0.7415 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/15

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Base Loss: 0.3535 | Tarreg Loss: 0.275780 | Total Loss: 0.6293 | Val Loss: 194.2418 | Raw Qini: 1.1679 | EMA Trend: 1.1679 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Base Loss: 0.3440 | Tarreg Loss: 0.264724 | Total Loss: 0.6088 | Val Loss: 194.1549 | Raw Qini: 1.1418 | EMA Trend: 1.1640 | (patience: 1/20)
Epoch 3/150 | Base Loss: 0.3537 | Tarreg Loss: 0.267491 | Total Loss: 0.6212 | Val Loss: 194.0542 | Raw Qini: 0.8297 | EMA Trend: 1.1138 | (patience: 2/20)
Epoch 4/150 | Base Loss: 15190.1523 | Tarreg Loss: 2543.465332 | Total Loss: 17733.6172 | Val Loss: 193.9240 | Raw Qini: 0.7820 | EMA Trend: 1.0641 | (patience: 3/20)
Epoch 5/150 | Base Loss: 0.3572 | Tarreg Loss: 0.224238 | Total Loss: 0.5814 | Val Loss: 193.7502 | Raw Qini: 0.8174 | EMA Trend: 1.0270 | (patience: 4/20)
Epoch 6/150 | Base Loss: 0.4901 | Tarreg Loss: 0.381318 | Total Loss: 0.8714 | Val Loss: 193.5506 | Raw Qini: 0.8525 | EMA Trend: 1.0009 | (patience: 5/20)
Epoch 7/150 | Base Loss: 0.5446 | Tarreg Los

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Revenue/Dragonnet/dragonnet.py:328: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
